# Watanabe concordance: Figure 1 and final scDRS-FM versus scDRS comparison

This trimmed notebook keeps only the analysis required to reproduce:

1. **Figure 1:** the Watanabe/scDRS-FM trait-by-cell-type association matrix;
2. **the final comparison figure from the source notebook:** exact three-way overlap plus Watanabe-enrichment odds ratios;
3. **the direct fold-enrichment comparison:** a one-sided paired trait bootstrap testing whether scDRS-FM has greater fold enrichment than scDRS; and
4. **manuscript-ready CSV source tables:** cell-type counts, reference associations, method calls, signal links, overlap membership, enrichment statistics, bootstrap replicates, and a file manifest.

All intermediate figures, broad audit tables, and alternative superiority tests remain removed. The cached figures below are retained from the source notebook; rerunning the notebook regenerates them from the configured inputs and writes the supplementary tables under `watanabe_unified_outputs/tables/csv/`.


## 1. Imports and configuration

The defaults reproduce the original project layout. Paths can instead be supplied through the environment variables documented below.


In [1]:
from __future__ import annotations

import os
import re
import warnings
from collections import defaultdict
from pathlib import Path
from typing import Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 100)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Output directory.
OUTPUT_DIR = Path(os.getenv("WATANABE_OUTPUT_DIR", "watanabe_unified_outputs"))
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CSV_DIR = TABLE_DIR / "csv"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

# TMS cell annotations. A flat table is the lightest-weight option; otherwise
# only .obs is read from the H5AD file.
BIOCOL = os.getenv("TMS_CELLTYPE_COLUMN", "cell_ontology_class")
CELL_ID_COLUMN = os.getenv("TMS_CELL_ID_COLUMN", "cell_id")
TMS_OBS_TABLE = Path(os.environ["TMS_OBS_TABLE"]) if os.getenv("TMS_OBS_TABLE") else None
TMS_H5AD_CANDIDATES = [
    Path(os.environ["TMS_H5AD_FILE"]) if os.getenv("TMS_H5AD_FILE") else None,
    Path("../scdrs_simpleby/tms_data/tabula_muris_senis/TMS_FACS.h5ad"),
    Path("tms_data/tabula_muris_senis/TMS_FACS.h5ad"),
]
TMS_H5AD_CANDIDATES = [path for path in TMS_H5AD_CANDIDATES if path is not None]

# scDRS-FM and scDRS result directories.
SCDRSFM_RESULTS_DIR_CANDIDATES = [
    Path(os.environ["SCDRSFM_RESULTS_DIR"]) if os.getenv("SCDRSFM_RESULTS_DIR") else None,
    Path("../scdrs_simpleby/august_all/scdrs+_results_gram3"),
    Path("august_all/scdrs+_results_gram3"),
]
SCDRSFM_RESULTS_DIR_CANDIDATES = [path for path in SCDRSFM_RESULTS_DIR_CANDIDATES if path is not None]

SCDRS_RESULTS_DIR_CANDIDATES = [
    Path(os.environ["SCDRS_RESULTS_DIR"]) if os.getenv("SCDRS_RESULTS_DIR") else None,
    Path("tms_data/scdrs_results"),
    Path("../scdrs_simpleby/tms_data/scdrs_results"),
]
SCDRS_RESULTS_DIR_CANDIDATES = [path for path in SCDRS_RESULTS_DIR_CANDIDATES if path is not None]

# Calling rules retained from the source notebook.
SCDRSFM_FDR = 0.10
SCDRS_FDR = 0.05
CELLTYPE_CALL_THRESHOLD = 0.05       # final calls require fraction > 0.05
SIGNAL_ASSOCIATION_THRESHOLD = 0.05 # signal/cell-type links require fraction >= 0.05
FILTER_CELLS_MIN_GENES = int(os.getenv("TMS_MIN_GENES_PER_CELL", "250"))

SCDRSFM_PVALUE_COLUMNS = ("pval", "mc_pval")
SCDRSFM_SIGNAL_COLUMN = "independent_signal_multi"
SCDRS_PVALUE_COLUMN = "assoc_mcp"

BOOTSTRAP_REPS = 20_000
BOOTSTRAP_SEED = 2026
FIGURE_DPI = 300

print(f"Working directory: {Path.cwd().resolve()}")
print(f"Output directory:  {OUTPUT_DIR.resolve()}")


Working directory: /workspace/nbwork/06_watanabe_overlap
Output directory:  /workspace/nbwork/06_watanabe_overlap/watanabe_unified_outputs


## 2. Watanabe reference associations

These are the ten traits and published TMS cell-type associations used by both retained figures and the enrichment comparison. `brain pericyte` is harmonized to the TMS label `pericyte cell` when that label exists in the annotation universe.


In [2]:
WATANABE_CELL_TYPES_RAW = {
    "PASS_Multiple_sclerosis": [
        "professional antigen presenting cell", "B cell", "macrophage"],
    "PASS_Schizophrenia_Pardinas2018": [
        "neuron", "interneuron", "medium spiny neuron"],
    "UKB_460K.body_BMIz": [
        "neuron", "interneuron", "medium spiny neuron",
        "pancreatic B cell", "oligodendrocyte precursor cell"],
    "PASS_Intelligence_SavageJansen2018": [
        "neuron", "interneuron", "medium spiny neuron", "pancreatic B cell"],
    "UKB_460K.mental_NEUROTICISM": [
        "neuron", "interneuron", "medium spiny neuron",
        "pancreatic PP cell", "pancreatic A cell"],
    "UKB_460K.bp_DIASTOLICadjMEDz": [
        "brain pericyte", "endothelial cell"],
    "UKB_460K.bp_SYSTOLICadjMEDz": [
        "brain pericyte", "fibroblast of cardiac tissue",
        "smooth muscle cell", "endothelial cell", "pancreatic stellate cell"],
    "PASS_Rheumatoid_Arthritis": [
        "B cell", "T cell", "mature NK T cell", "regulatory T cell",
        "immature T cell", "professional antigen presenting cell", "lymphocyte"],
    "PASS_Type_1_Diabetes": ["B cell", "T cell"],
    "PASS_Lupus": [
        "B cell", "lymphocyte", "naive B cell", "myeloid cell",
        "leukocyte", "professional antigen presenting cell"],
}

TRAIT_LABELS = {
    "PASS_Multiple_sclerosis": "Multiple sclerosis",
    "PASS_Schizophrenia_Pardinas2018": "Schizophrenia",
    "UKB_460K.body_BMIz": "BMI",
    "PASS_Intelligence_SavageJansen2018": "Intelligence",
    "UKB_460K.mental_NEUROTICISM": "Neuroticism",
    "UKB_460K.bp_DIASTOLICadjMEDz": "Diastolic blood pressure",
    "UKB_460K.bp_SYSTOLICadjMEDz": "Systolic blood pressure",
    "PASS_Rheumatoid_Arthritis": "Rheumatoid arthritis",
    "PASS_Type_1_Diabetes": "Type 1 diabetes",
    "PASS_Lupus": "Lupus",
}

TRAITS = list(WATANABE_CELL_TYPES_RAW)
CELLTYPE_ALIASES = {"brain pericyte": "pericyte cell"}

print(f"Analysis set: {len(TRAITS)} traits and "
      f"{sum(map(len, WATANABE_CELL_TYPES_RAW.values()))} reference associations.")


Analysis set: 10 traits and 42 reference associations.


## 3. Shared helpers and TMS annotation loading

Only cell IDs and cell-type annotations are required. Expression values are not loaded into memory.


In [3]:
def first_existing_path(candidates: Sequence[Path]) -> Path | None:
    return next((path for path in candidates if path.exists()), None)


def resolve_required_directory(candidates: Sequence[Path], label: str) -> Path:
    path = first_existing_path(candidates)
    if path is None:
        checked = "\n".join(f"  - {candidate}" for candidate in candidates)
        raise FileNotFoundError(f"Could not find {label}. Checked:\n{checked}")
    return path


def normalize_label(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def unique_strings(values: Iterable[object]) -> list[str]:
    return list(dict.fromkeys(
        str(value).strip()
        for value in values
        if pd.notna(value) and str(value).strip()
    ))


def pick_column(
    frame: pd.DataFrame,
    candidates: Sequence[str],
    *,
    description: str,
    required: bool = True,
) -> str | None:
    exact = {str(column): str(column) for column in frame.columns}
    casefolded = {str(column).strip().lower(): str(column) for column in frame.columns}
    normalized = {normalize_label(column): str(column) for column in frame.columns}

    for candidate in candidates:
        if candidate in exact:
            return exact[candidate]
        if candidate.strip().lower() in casefolded:
            return casefolded[candidate.strip().lower()]
        if normalize_label(candidate) in normalized:
            return normalized[normalize_label(candidate)]

    if required:
        raise ValueError(
            f"Could not find {description}; checked {list(candidates)}. "
            f"Available columns: {frame.columns.tolist()}"
        )
    return None


def bh_qvalues(pvalues: Iterable[object]) -> np.ndarray:
    values = pd.to_numeric(pd.Series(list(pvalues)), errors="coerce").to_numpy(dtype=float)
    qvalues = np.full(len(values), np.nan, dtype=float)
    valid = np.isfinite(values)
    if valid.any():
        qvalues[valid] = multipletests(values[valid], method="fdr_bh")[1]
    return qvalues


def read_table_auto(path: Path, **kwargs) -> pd.DataFrame:
    name = path.name.lower()
    tab_delimited = any(token in name for token in (
        ".tsv", ".txt", ".score", ".cell_ontology_class"
    ))
    return pd.read_csv(
        path,
        sep="\t" if tab_delimited else ",",
        compression="infer",
        **kwargs,
    )


def save_table(
    frame: pd.DataFrame,
    relative_path: str,
    *,
    also_csv: bool = True,
    **kwargs,
) -> Path:
    """Save a table and, by default, a comma-delimited manuscript CSV mirror.

    TSV outputs retained from the source notebook are written unchanged. Their
    CSV mirrors are placed in ``TABLE_DIR / "csv"`` with the same stem.
    """
    path = OUTPUT_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, **kwargs)
    print(f"Saved: {path}")

    if also_csv and path.suffix.lower() != ".csv":
        try:
            relative_to_tables = path.relative_to(TABLE_DIR)
        except ValueError:
            relative_to_tables = Path(path.name)

        csv_path = CSV_DIR / relative_to_tables.with_suffix(".csv")
        csv_path.parent.mkdir(parents=True, exist_ok=True)

        csv_kwargs = dict(kwargs)
        csv_kwargs.pop("sep", None)
        csv_kwargs.pop("compression", None)
        frame.to_csv(csv_path, sep=",", **csv_kwargs)
        print(f"Saved CSV mirror: {csv_path}")

    return path


def split_cell_ids(values: Iterable[object]) -> pd.Index:
    cell_ids: list[str] = []
    for value in values:
        if pd.isna(value):
            continue
        cell_ids.extend(token for token in re.split(r"[,;\s]+", str(value).strip()) if token)
    return pd.Index(unique_strings(cell_ids), dtype=str)


def load_tms_obs() -> tuple[pd.DataFrame, str]:
    if TMS_OBS_TABLE is not None:
        if not TMS_OBS_TABLE.exists():
            raise FileNotFoundError(f"TMS_OBS_TABLE does not exist: {TMS_OBS_TABLE}")
        obs = read_table_auto(TMS_OBS_TABLE)
        if CELL_ID_COLUMN in obs.columns:
            obs = obs.set_index(CELL_ID_COLUMN)
        elif str(obs.columns[0]).lower().startswith("unnamed"):
            obs = obs.set_index(obs.columns[0])
        elif obs.index.equals(pd.RangeIndex(len(obs))):
            raise ValueError(
                f"{TMS_OBS_TABLE} needs a {CELL_ID_COLUMN!r} column or cell IDs as its index."
            )
        source = str(TMS_OBS_TABLE)
    else:
        h5ad_path = first_existing_path(TMS_H5AD_CANDIDATES)
        if h5ad_path is None:
            raise FileNotFoundError(
                "No TMS annotation source found. Set TMS_OBS_TABLE or TMS_H5AD_FILE."
            )
        try:
            import anndata as ad
        except ImportError as exc:
            raise ImportError(
                "Reading the H5AD requires `anndata`; alternatively set TMS_OBS_TABLE."
            ) from exc

        adata = ad.read_h5ad(h5ad_path, backed="r")
        obs = adata.obs.copy()
        obs.index = pd.Index(adata.obs_names.astype(str), dtype=str)
        try:
            adata.file.close()
        except Exception:
            pass
        source = str(h5ad_path)

    if BIOCOL not in obs.columns:
        raise ValueError(f"TMS annotations are missing {BIOCOL!r}.")

    obs = obs.copy()
    obs.index = pd.Index(obs.index.astype(str), dtype=str)
    if obs.index.has_duplicates:
        raise ValueError("TMS cell IDs must be unique.")

    obs[BIOCOL] = obs[BIOCOL].astype("string")
    obs = obs.loc[obs[BIOCOL].notna() & obs[BIOCOL].str.strip().ne("")]
    obs[BIOCOL] = obs[BIOCOL].astype(str)

    gene_count_column = pick_column(
        obs,
        ("n_genes", "n_genes_by_counts", "nGene", "nFeature_RNA"),
        description="per-cell gene count",
        required=False,
    )
    if FILTER_CELLS_MIN_GENES > 0 and gene_count_column is not None:
        before = len(obs)
        keep = pd.to_numeric(obs[gene_count_column], errors="coerce").fillna(0).ge(
            FILTER_CELLS_MIN_GENES
        )
        obs = obs.loc[keep]
        print(
            f"Applied {gene_count_column} >= {FILTER_CELLS_MIN_GENES}: "
            f"{before:,} -> {len(obs):,} cells"
        )
    elif FILTER_CELLS_MIN_GENES > 0:
        warnings.warn(
            "No n_genes-like annotation column was found; all annotated cells are retained."
        )

    return obs, source


obs, TMS_ANNOTATION_SOURCE = load_tms_obs()
CELLTYPE_COUNTS = obs[BIOCOL].value_counts().sort_index()
CELLTYPE_UNIVERSE = sorted(CELLTYPE_COUNTS.index.astype(str))
CELL_ID_UNIVERSE = pd.Index(obs.index.astype(str), dtype=str)

print(f"TMS annotations: {TMS_ANNOTATION_SOURCE}")
print(f"Loaded {len(obs):,} cells across {len(CELLTYPE_UNIVERSE):,} cell types.")


Applied n_genes >= 250: 10,000 -> 10,000 cells
TMS annotations: /mnt/shared-workspace/scdrsfm/data/subsets_10k/TMS_FACS/TMS_FACS.h5ad
Loaded 10,000 cells across 116 cell types.


## 4. Harmonize the Watanabe labels to the TMS universe

Mapped labels are used in fixed-universe statistics. The display dictionary also retains any unmatched Watanabe label so Figure 1 does not silently omit a published association.


In [4]:
def map_celltype_label(
    raw_label: object,
    available_labels: Sequence[str],
    aliases: Mapping[str, str] | None = None,
) -> tuple[str, bool]:
    aliases = aliases or {}
    raw = str(raw_label).strip()
    available = [str(label) for label in available_labels]
    available_set = set(available)
    normalized_map = {normalize_label(label): label for label in available}

    alias = aliases.get(raw, aliases.get(raw.lower()))
    for candidate in ([alias, raw] if alias else [raw]):
        if candidate in available_set:
            return candidate, True
        normalized = normalize_label(candidate)
        if normalized in normalized_map:
            return normalized_map[normalized], True
    return raw, False


def map_watanabe_dictionary(
    raw_dictionary: Mapping[str, Sequence[str]],
    available_labels: Sequence[str],
) -> tuple[dict[str, list[str]], dict[str, list[str]]]:
    display_dictionary: dict[str, list[str]] = {}
    analysis_dictionary: dict[str, list[str]] = {}

    for trait, raw_labels in raw_dictionary.items():
        display_labels: list[str] = []
        analysis_labels: list[str] = []
        for raw_label in raw_labels:
            mapped, in_universe = map_celltype_label(
                raw_label,
                available_labels,
                CELLTYPE_ALIASES,
            )
            display_labels.append(mapped)
            if in_universe:
                analysis_labels.append(mapped)
        display_dictionary[trait] = unique_strings(display_labels)
        analysis_dictionary[trait] = unique_strings(analysis_labels)

    return display_dictionary, analysis_dictionary


WATANABE_DISPLAY_BY_TRAIT, WATANABE_MAPPED_BY_TRAIT = map_watanabe_dictionary(
    WATANABE_CELL_TYPES_RAW,
    CELLTYPE_UNIVERSE,
)


## 5. Call scDRS-FM cell types and independent-signal links

For each trait:

- marginal cells are called at BH-FDR 0.10;
- conditionally significant metacells are called at BH-FDR 0.10;
- the retained cells are their intersection; and
- a cell type is called when **more than 5%** of all TMS cells of that type are retained.

Signal-sharing links use the source notebook's `>= 5%` signal-specific fraction rule.


In [5]:
def discovered_fraction_by_celltype(cell_ids: Iterable[object]) -> pd.Series:
    ids = CELL_ID_UNIVERSE.intersection(pd.Index([str(value) for value in cell_ids], dtype=str))
    if len(ids) == 0:
        return pd.Series(0.0, index=CELLTYPE_UNIVERSE, dtype=float)
    discovered_counts = obs.loc[ids, BIOCOL].value_counts()
    return (
        discovered_counts
        .div(CELLTYPE_COUNTS)
        .reindex(CELLTYPE_UNIVERSE, fill_value=0.0)
        .astype(float)
    )


def load_scdrsfm_calls(
    result_dir: Path,
    traits: Sequence[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    missing = [
        result_dir / f"{trait}{suffix}"
        for trait in traits
        for suffix in (".marginal_score.gz", ".conditional.tagging_score.gz")
        if not (result_dir / f"{trait}{suffix}").exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing scDRS-FM inputs:\n" + "\n".join(f"  - {path}" for path in missing)
        )

    fraction_rows: dict[str, pd.Series] = {}
    signal_rows: list[dict[str, object]] = []

    for trait in traits:
        marginal_path = result_dir / f"{trait}.marginal_score.gz"
        conditional_path = result_dir / f"{trait}.conditional.tagging_score.gz"

        marginal = pd.read_csv(marginal_path, sep="\t", compression="gzip", index_col=0)
        marginal_p_column = pick_column(
            marginal,
            SCDRSFM_PVALUE_COLUMNS,
            description=f"marginal p-value in {marginal_path}",
        )
        marginal_qvalues = bh_qvalues(marginal[marginal_p_column])
        marginal_cells = CELL_ID_UNIVERSE.intersection(pd.Index(
            marginal.index[
                np.isfinite(marginal_qvalues) & (marginal_qvalues <= SCDRSFM_FDR)
            ].astype(str),
            dtype=str,
        ))

        conditional = pd.read_csv(
            conditional_path,
            sep="\t",
            compression="gzip",
            index_col=0,
        )
        for required_column in ("cell_ids", SCDRSFM_SIGNAL_COLUMN):
            if required_column not in conditional.columns:
                raise ValueError(f"{conditional_path} is missing {required_column!r}.")

        conditional_p_column = pick_column(
            conditional,
            SCDRSFM_PVALUE_COLUMNS,
            description=f"conditional p-value in {conditional_path}",
        )
        conditional_qvalues = bh_qvalues(conditional[conditional_p_column])
        conditional_significant = conditional.loc[
            np.isfinite(conditional_qvalues) & (conditional_qvalues <= SCDRSFM_FDR)
        ].copy()

        conditional_cells = CELL_ID_UNIVERSE.intersection(
            split_cell_ids(conditional_significant["cell_ids"])
            if len(conditional_significant)
            else pd.Index([], dtype=str)
        )
        retained_cells = marginal_cells.intersection(conditional_cells)
        fraction_rows[trait] = discovered_fraction_by_celltype(retained_cells)

        # Associate nonnegative independent signals with cell types using the
        # signal-specific marginal x conditional cell fractions.
        signal_values = pd.to_numeric(
            conditional_significant[SCDRSFM_SIGNAL_COLUMN],
            errors="coerce",
        )
        signal_ids = sorted(
            signal_values[signal_values.notna() & (signal_values >= 0)]
            .astype(int)
            .unique()
        )
        for signal_id in signal_ids:
            signal_mask = signal_values.astype("Int64").eq(signal_id).fillna(False)
            signal_cells = CELL_ID_UNIVERSE.intersection(
                split_cell_ids(conditional_significant.loc[signal_mask, "cell_ids"])
            )
            signal_retained_cells = marginal_cells.intersection(signal_cells)
            fractions = discovered_fraction_by_celltype(signal_retained_cells)

            for cell_type, fraction in fractions.items():
                if float(fraction) >= SIGNAL_ASSOCIATION_THRESHOLD:
                    signal_rows.append({
                        "trait": trait,
                        "cell_type": cell_type,
                        "independent_signal": int(signal_id),
                        "signal_discovery_fraction": float(fraction),
                    })

        print(
            f"[{TRAIT_LABELS[trait]}] marginal={len(marginal_cells):,}; "
            f"conditional={len(conditional_cells):,}; intersection={len(retained_cells):,}"
        )

    fractions = pd.DataFrame.from_dict(fraction_rows, orient="index").reindex(traits)
    fractions.index.name = "trait"
    fractions.columns.name = BIOCOL
    signal_associations = pd.DataFrame(
        signal_rows,
        columns=[
            "trait", "cell_type", "independent_signal", "signal_discovery_fraction"
        ],
    )
    return fractions, signal_associations


SCDRSFM_RESULTS_DIR = resolve_required_directory(
    SCDRSFM_RESULTS_DIR_CANDIDATES,
    "the scDRS-FM result directory",
)
print(f"scDRS-FM directory: {SCDRSFM_RESULTS_DIR.resolve()}")

(
    scdrsfm_marginal_x_conditional_fractions,
    scdrsfm_signal_celltype_associations,
) = load_scdrsfm_calls(SCDRSFM_RESULTS_DIR, TRAITS)

# The source notebook's retained figures use a strict >5% call threshold.
SCDRSFM_BY_TRAIT = {
    trait: scdrsfm_marginal_x_conditional_fractions.columns[
        scdrsfm_marginal_x_conditional_fractions.loc[trait].gt(CELLTYPE_CALL_THRESHOLD)
    ].astype(str).tolist()
    for trait in TRAITS
}

SCDRSFM_SIGNALS_BY_TRAIT_CELLTYPE: dict[str, dict[str, set[int]]] = {
    trait: defaultdict(set) for trait in TRAITS
}
for row in scdrsfm_signal_celltype_associations.itertuples(index=False):
    SCDRSFM_SIGNALS_BY_TRAIT_CELLTYPE[str(row.trait)][str(row.cell_type)].add(
        int(row.independent_signal)
    )


scDRS-FM directory: /mnt/shared-workspace/scdrsfm/results/real/tms_facs


[Multiple sclerosis] marginal=2,564; conditional=577; intersection=576


[Schizophrenia] marginal=212; conditional=89; intersection=89


[BMI] marginal=329; conditional=148; intersection=140


[Intelligence] marginal=410; conditional=245; intersection=159


...

[Diastolic blood pressure] marginal=1,070; conditional=300; intersection=257


[Systolic blood pressure] marginal=365; conditional=124; intersection=111


[Rheumatoid arthritis] marginal=1,717; conditional=562; intersection=546


[Type 1 diabetes] marginal=562; conditional=410; intersection=377


[Lupus] marginal=1,357; conditional=945; intersection=870


## 6. Figure 1

The marker shape/color encodes direct Watanabe overlap, scDRS-FM-only calls, Watanabe-only calls, and scDRS-FM-only calls that share an independent signal with a direct Watanabe-overlap cell type for the same trait.


In [6]:
def build_signal_sharing_detail() -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for trait in TRAITS:
        fm_types = set(SCDRSFM_BY_TRAIT.get(trait, []))
        watanabe_types = set(WATANABE_MAPPED_BY_TRAIT.get(trait, []))
        direct_overlap = fm_types & watanabe_types
        signal_map = SCDRSFM_SIGNALS_BY_TRAIT_CELLTYPE.get(trait, {})

        anchor_signals: set[int] = set()
        for cell_type in direct_overlap:
            anchor_signals.update(signal_map.get(cell_type, set()))

        for cell_type in sorted(fm_types):
            signals = set(signal_map.get(cell_type, set()))
            shared_signals = signals & anchor_signals
            rows.append({
                "trait": trait,
                "cell_type": cell_type,
                "is_direct_watanabe_overlap": cell_type in direct_overlap,
                "independent_signals": ";".join(map(str, sorted(signals))),
                "shares_signal_with_direct_watanabe_overlap": bool(shared_signals),
                "shared_watanabe_signal_ids": ";".join(map(str, sorted(shared_signals))),
            })
    return pd.DataFrame(rows)


def build_figure1_table() -> pd.DataFrame:
    signal_detail = build_signal_sharing_detail()
    signal_lookup = {
        (row.trait, row.cell_type): row
        for row in signal_detail.itertuples(index=False)
    }

    rows: list[dict[str, object]] = []
    for trait in TRAITS:
        watanabe_display = set(WATANABE_DISPLAY_BY_TRAIT.get(trait, []))
        watanabe_mapped = set(WATANABE_MAPPED_BY_TRAIT.get(trait, []))
        fm_types = set(SCDRSFM_BY_TRAIT.get(trait, []))

        for cell_type in sorted(watanabe_display | fm_types):
            is_watanabe = cell_type in watanabe_display
            is_watanabe_mapped = cell_type in watanabe_mapped
            is_scdrsfm = cell_type in fm_types
            direct_overlap = is_scdrsfm and is_watanabe_mapped

            signal_row = signal_lookup.get((trait, cell_type))
            shares_signal = bool(
                signal_row.shares_signal_with_direct_watanabe_overlap
            ) if signal_row else False

            if direct_overlap:
                status = "Direct Watanabe + scDRS-FM overlap"
            elif is_scdrsfm and shares_signal:
                status = "scDRS-FM only; shares Watanabe-overlap signal"
            elif is_scdrsfm:
                status = "scDRS-FM only"
            else:
                status = "Watanabe only"

            rows.append({
                "trait": trait,
                "trait_label": TRAIT_LABELS[trait],
                "cell_type": cell_type,
                "watanabe_listed": is_watanabe,
                "watanabe_mapped_to_tms": is_watanabe_mapped,
                "scdrsfm_associated": is_scdrsfm,
                "direct_overlap": direct_overlap,
                "shares_signal_with_direct_overlap": shares_signal,
                "scdrsfm_discovery_fraction": (
                    float(scdrsfm_marginal_x_conditional_fractions.loc[trait, cell_type])
                    if is_scdrsfm else np.nan
                ),
                "independent_signals": (
                    str(signal_row.independent_signals) if signal_row else ""
                ),
                "shared_watanabe_signal_ids": (
                    str(signal_row.shared_watanabe_signal_ids) if signal_row else ""
                ),
                "status": status,
            })

    return pd.DataFrame(rows)


def plot_figure1(plot_frame: pd.DataFrame) -> tuple[plt.Figure, plt.Axes]:
    watanabe_first: list[str] = []
    for trait in TRAITS:
        for cell_type in WATANABE_DISPLAY_BY_TRAIT[trait]:
            if cell_type not in watanabe_first:
                watanabe_first.append(cell_type)

    extra_cell_types = sorted(
        set(plot_frame["cell_type"]) - set(watanabe_first),
        key=lambda cell_type: (
            -int((plot_frame["cell_type"] == cell_type).sum()),
            cell_type.lower(),
        ),
    )
    celltype_order = watanabe_first + extra_cell_types
    x_position = {trait: index for index, trait in enumerate(TRAITS)}
    y_position = {cell_type: index for index, cell_type in enumerate(celltype_order)}

    styles = {
        "Direct Watanabe + scDRS-FM overlap": {
            "marker": "D", "color": "#2a9d8f", "edgecolor": "#1f6f68"},
        "scDRS-FM only; shares Watanabe-overlap signal": {
            "marker": "s", "color": "#f4a261", "edgecolor": "#b45f06"},
        "scDRS-FM only": {
            "marker": "o", "color": "#457b9d", "edgecolor": "#274c77"},
        "Watanabe only": {
            "marker": "X", "color": "#8d99ae", "edgecolor": "#5c677d"},
    }
    legend_labels = {
        "Direct Watanabe + scDRS-FM overlap": "Watanabe + scDRS-FM",
        "scDRS-FM only; shares Watanabe-overlap signal": "scDRS-FM with shared signal",
        "scDRS-FM only": "scDRS-FM only",
        "Watanabe only": "Watanabe only",
    }

    side = max(13.0, min(18.0, 0.38 * len(celltype_order) + 4.0))
    figure, axis = plt.subplots(figsize=(side, side))

    for status, style in styles.items():
        subset = plot_frame.loc[plot_frame["status"].eq(status)]
        if subset.empty:
            continue
        axis.scatter(
            [x_position[trait] for trait in subset["trait"]],
            [y_position[cell_type] for cell_type in subset["cell_type"]],
            s=190,
            marker=style["marker"],
            c=style["color"],
            edgecolors=style["edgecolor"],
            linewidths=1.35,
            alpha=0.95,
            zorder=3,
        )

    axis.set_xticks(range(len(TRAITS)))
    axis.set_xticklabels(
        [TRAIT_LABELS[trait] for trait in TRAITS],
        rotation=42,
        ha="right",
        fontsize=14,
    )
    axis.set_yticks(range(len(celltype_order)))
    axis.set_yticklabels(celltype_order, fontsize=12.5)
    axis.tick_params(axis="both", which="major", length=0, pad=7)
    axis.invert_yaxis()
    axis.set_xlim(-0.6, len(TRAITS) - 0.35)
    axis.set_ylim(len(celltype_order) - 0.4, -0.6)
    axis.set_axisbelow(True)
    axis.grid(True, which="major", color="#e9ecef", linewidth=0.8)

    handles = [
        Line2D(
            [0], [0],
            marker=style["marker"],
            color="none",
            markerfacecolor=style["color"],
            markeredgecolor=style["edgecolor"],
            markeredgewidth=1.35,
            markersize=11,
            label=legend_labels[status],
        )
        for status, style in styles.items()
    ]
    axis.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.20),
        ncol=2,
        frameon=False,
        fontsize=13,
        handletextpad=0.7,
        columnspacing=1.5,
    )

    figure.tight_layout(rect=(0, 0.10, 1, 1))
    return figure, axis


figure1_table = build_figure1_table()
save_table(
    figure1_table,
    "tables/figure1_scdrsfm_watanabe_celltype_signal_overlap.tsv",
    sep="\t",
    index=False,
)

figure1, _ = plot_figure1(figure1_table)
figure1_png = FIGURE_DIR / "figure1_scdrsfm_watanabe_celltype_signal_overlap.png"
figure1_pdf = FIGURE_DIR / "figure1_scdrsfm_watanabe_celltype_signal_overlap.pdf"
figure1.savefig(figure1_png, dpi=FIGURE_DPI, bbox_inches="tight")
figure1.savefig(figure1_pdf, bbox_inches="tight")
plt.show()
print(f"Saved: {figure1_png}\nSaved: {figure1_pdf}")


Saved: watanabe_unified_outputs/tables/figure1_scdrsfm_watanabe_celltype_signal_overlap.tsv
Saved CSV mirror: watanabe_unified_outputs/tables/csv/figure1_scdrsfm_watanabe_celltype_signal_overlap.csv


Saved: watanabe_unified_outputs/figures/figure1_scdrsfm_watanabe_celltype_signal_overlap.png
Saved: watanabe_unified_outputs/figures/figure1_scdrsfm_watanabe_celltype_signal_overlap.pdf


## 7. Load scDRS calls

Each `{trait}.scdrs_ct.cell_ontology_class` file is corrected across cell types with Benjamini-Hochberg FDR at 0.05. Only mapped TMS cell types enter the fixed comparison universe.


In [7]:
def load_scdrs_calls(
    result_dir: Path,
    traits: Sequence[str],
) -> dict[str, list[str]]:
    missing = [
        result_dir / f"{trait}.scdrs_ct.{BIOCOL}"
        for trait in traits
        if not (result_dir / f"{trait}.scdrs_ct.{BIOCOL}").exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Missing scDRS inputs:\n" + "\n".join(f"  - {path}" for path in missing)
        )

    calls_by_trait: dict[str, list[str]] = {}
    for trait in traits:
        path = result_dir / f"{trait}.scdrs_ct.{BIOCOL}"
        table = read_table_auto(path, index_col=0)
        if SCDRS_PVALUE_COLUMN not in table.columns:
            raise ValueError(
                f"{path} is missing {SCDRS_PVALUE_COLUMN!r}. "
                f"Available columns: {table.columns.tolist()}"
            )

        qvalues = bh_qvalues(table[SCDRS_PVALUE_COLUMN])
        significant_labels = table.index[
            np.isfinite(qvalues) & (qvalues <= SCDRS_FDR)
        ].astype(str)

        mapped_calls: set[str] = set()
        for label in significant_labels:
            mapped, in_universe = map_celltype_label(label, CELLTYPE_UNIVERSE)
            if in_universe:
                mapped_calls.add(mapped)
        calls_by_trait[trait] = sorted(mapped_calls)

    return calls_by_trait


SCDRS_RESULTS_DIR = resolve_required_directory(
    SCDRS_RESULTS_DIR_CANDIDATES,
    "the scDRS result directory",
)
print(f"scDRS directory: {SCDRS_RESULTS_DIR.resolve()}")
SCDRS_BY_TRAIT = load_scdrs_calls(SCDRS_RESULTS_DIR, TRAITS)


scDRS directory: /mnt/shared-workspace/scdrsfm/results/ct/tms_facs_none_ctrl


## 8. Fixed-universe enrichment and direct fold-enrichment comparison

The comparison universe is all `10 traits × all annotated TMS cell types`. For each method, fold enrichment is the observed Watanabe overlap divided by the overlap expected under independence.

The direct scDRS-FM versus scDRS p-value comes from a **paired trait bootstrap**: each replicate samples ten complete traits with replacement, preserving every cell-type call within a sampled trait. The alternative is `fold enrichment(scDRS-FM) > fold enrichment(scDRS)`.


In [8]:
CELLTYPE_SET = set(CELLTYPE_UNIVERSE)
N_CELL_TYPES = len(CELLTYPE_SET)
if N_CELL_TYPES == 0:
    raise ValueError("The TMS cell-type universe is empty.")

watanabe_by_trait = {
    trait: set(WATANABE_MAPPED_BY_TRAIT.get(trait, [])) & CELLTYPE_SET
    for trait in TRAITS
}
scdrsfm_by_trait = {
    trait: set(SCDRSFM_BY_TRAIT.get(trait, [])) & CELLTYPE_SET
    for trait in TRAITS
}
scdrs_by_trait = {
    trait: set(SCDRS_BY_TRAIT.get(trait, [])) & CELLTYPE_SET
    for trait in TRAITS
}


def pair_set(calls_by_trait: Mapping[str, Iterable[str]]) -> set[tuple[str, str]]:
    return {
        (trait, cell_type)
        for trait in TRAITS
        for cell_type in calls_by_trait.get(trait, [])
    }


watanabe_pairs = pair_set(watanabe_by_trait)
scdrsfm_pairs = pair_set(scdrsfm_by_trait)
scdrs_pairs = pair_set(scdrs_by_trait)

exact_regions = {
    "Watanabe only": len(watanabe_pairs - scdrsfm_pairs - scdrs_pairs),
    "scDRS-FM only": len(scdrsfm_pairs - watanabe_pairs - scdrs_pairs),
    "scDRS only": len(scdrs_pairs - watanabe_pairs - scdrsfm_pairs),
    "Watanabe + scDRS-FM only": len((watanabe_pairs & scdrsfm_pairs) - scdrs_pairs),
    "Watanabe + scDRS only": len((watanabe_pairs & scdrs_pairs) - scdrsfm_pairs),
    "scDRS-FM + scDRS only": len((scdrsfm_pairs & scdrs_pairs) - watanabe_pairs),
    "All three": len(watanabe_pairs & scdrsfm_pairs & scdrs_pairs),
}


def contingency_counts(
    method_calls: set[str],
    reference_calls: set[str],
) -> np.ndarray:
    overlap = len(method_calls & reference_calls)
    method_only = len(method_calls - reference_calls)
    reference_only = len(reference_calls - method_calls)
    neither = N_CELL_TYPES - overlap - method_only - reference_only
    if neither < 0:
        raise AssertionError("Calls must be restricted to CELLTYPE_UNIVERSE.")
    return np.asarray([overlap, method_only, reference_only, neither], dtype=int)


def finite_odds_ratio_ci(counts: np.ndarray) -> tuple[float, float, float, float]:
    cells = np.asarray(counts, dtype=float).copy()
    if np.any(cells == 0):
        cells += 0.5
    a, b, c, d = cells
    log_or = np.log((a * d) / (b * c))
    standard_error = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    return (
        float(np.exp(log_or)),
        float(np.exp(log_or - 1.96 * standard_error)),
        float(np.exp(log_or + 1.96 * standard_error)),
        float(log_or),
    )


def summarize_enrichment(method: str, counts: np.ndarray) -> dict[str, object]:
    a, b, c, d = map(int, counts)
    total = a + b + c + d
    method_calls = a + b
    reference_calls = a + c
    expected = method_calls * reference_calls / total if total else np.nan
    fisher_or, fisher_p = fisher_exact([[a, b], [c, d]], alternative="greater")
    finite_or, ci_low, ci_high, log_or = finite_odds_ratio_ci(counts)

    return {
        "method": method,
        "overlap": a,
        "method_calls": method_calls,
        "watanabe_calls": reference_calls,
        "method_only": b,
        "watanabe_only": c,
        "neither": d,
        "expected_overlap": expected,
        "fold_enrichment": a / expected if expected > 0 else np.nan,
        "fisher_odds_ratio": float(fisher_or),
        "continuity_corrected_odds_ratio": finite_or,
        "odds_ratio_ci95_low": ci_low,
        "odds_ratio_ci95_high": ci_high,
        "log_odds_ratio": log_or,
        "fisher_one_sided_enrichment_pvalue": float(fisher_p),
    }


counts_by_method_trait = {
    "scDRS-FM": {
        trait: contingency_counts(scdrsfm_by_trait[trait], watanabe_by_trait[trait])
        for trait in TRAITS
    },
    "scDRS": {
        trait: contingency_counts(scdrs_by_trait[trait], watanabe_by_trait[trait])
        for trait in TRAITS
    },
}

observed_counts = {
    method: np.vstack([trait_counts[trait] for trait in TRAITS]).sum(axis=0)
    for method, trait_counts in counts_by_method_trait.items()
}

method_watanabe_enrichment = pd.DataFrame([
    summarize_enrichment(method, counts)
    for method, counts in observed_counts.items()
])


def fold_enrichment_from_counts(count_matrix: np.ndarray) -> np.ndarray:
    counts = np.asarray(count_matrix, dtype=float)
    if counts.ndim == 1:
        counts = counts[None, :]
    a, b, c, d = counts.T
    method_calls = a + b
    reference_calls = a + c
    total = a + b + c + d
    expected = np.divide(
        method_calls * reference_calls,
        total,
        out=np.full_like(total, np.nan),
        where=total > 0,
    )
    return np.divide(
        a,
        expected,
        out=np.zeros_like(a),
        where=expected > 0,
    )


rng = np.random.default_rng(BOOTSTRAP_SEED)
trait_draws = rng.integers(
    low=0,
    high=len(TRAITS),
    size=(BOOTSTRAP_REPS, len(TRAITS)),
)

fm_trait_counts = np.vstack([
    counts_by_method_trait["scDRS-FM"][trait] for trait in TRAITS
])
scdrs_trait_counts = np.vstack([
    counts_by_method_trait["scDRS"][trait] for trait in TRAITS
])

fm_bootstrap_fold = fold_enrichment_from_counts(
    fm_trait_counts[trait_draws].sum(axis=1)
)
scdrs_bootstrap_fold = fold_enrichment_from_counts(
    scdrs_trait_counts[trait_draws].sum(axis=1)
)
bootstrap_difference = fm_bootstrap_fold - scdrs_bootstrap_fold

fm_observed_fold = float(fold_enrichment_from_counts(observed_counts["scDRS-FM"])[0])
scdrs_observed_fold = float(fold_enrichment_from_counts(observed_counts["scDRS"])[0])
observed_difference = fm_observed_fold - scdrs_observed_fold
ci_low, ci_high = np.quantile(bootstrap_difference, [0.025, 0.975])
one_sided_pvalue = (
    np.sum(bootstrap_difference <= 0.0) + 1
) / (BOOTSTRAP_REPS + 1)

fold_enrichment_comparison = pd.DataFrame([{
    "comparison": "scDRS-FM > scDRS",
    "scDRS-FM_fold_enrichment": fm_observed_fold,
    "scDRS_fold_enrichment": scdrs_observed_fold,
    "observed_difference": observed_difference,
    "bootstrap_ci95_low": float(ci_low),
    "bootstrap_ci95_high": float(ci_high),
    "bootstrap_probability_scDRSFM_greater": float(
        np.mean(bootstrap_difference > 0.0)
    ),
    "one_sided_trait_bootstrap_pvalue": float(one_sided_pvalue),
    "n_bootstrap_replicates": BOOTSTRAP_REPS,
    "n_traits_resampled_per_replicate": len(TRAITS),
    "seed": BOOTSTRAP_SEED,
}])

save_table(
    fold_enrichment_comparison,
    "tables/fold_enrichment_scdrsfm_vs_scdrs_trait_bootstrap.tsv",
    sep="\t",
    index=False,
)
display(fold_enrichment_comparison)


Saved: watanabe_unified_outputs/tables/fold_enrichment_scdrsfm_vs_scdrs_trait_bootstrap.tsv
Saved CSV mirror: watanabe_unified_outputs/tables/csv/fold_enrichment_scdrsfm_vs_scdrs_trait_bootstrap.csv


,comparison,scDRS-FM_fold_enrichment,scDRS_fold_enrichment,observed_difference,bootstrap_ci95_low,bootstrap_ci95_high,bootstrap_probability_scDRSFM_greater,one_sided_trait_bootstrap_pvalue,n_bootstrap_replicates,n_traits_resampled_per_replicate,seed
0,scDRS-FM > scDRS,8.083624,7.957317,0.126307,-3.548516,2.401817,0.5358,0.464227,20000,10,2026


## 9. Final comparison figure

This is the last plotted figure from the source notebook: exact exclusive overlap counts on the left and method-specific Watanabe-enrichment odds ratios on the right. The right panel also reports each method's Fisher enrichment p-value and fold enrichment.


In [9]:
def plot_final_comparison() -> plt.Figure:
    figure, (overlap_axis, odds_ratio_axis) = plt.subplots(
        1,
        2,
        figsize=(15, 7.5),
        gridspec_kw={"width_ratios": [1.15, 0.85]},
    )

    circle_specs = [
        ((-0.48, 0.22), "#8d99ae", "#5c677d"),
        ((0.48, 0.22), "#2a9d8f", "#1f6f68"),
        ((0.00, -0.52), "#457b9d", "#274c77"),
    ]
    for center, facecolor, edgecolor in circle_specs:
        overlap_axis.add_patch(Circle(
            center,
            radius=1.0,
            facecolor=facecolor,
            edgecolor=edgecolor,
            linewidth=2.5,
            alpha=0.27,
        ))

    count_positions = {
        "Watanabe only": (-0.98, 0.45),
        "scDRS-FM only": (0.98, 0.45),
        "scDRS only": (0.00, -1.18),
        "Watanabe + scDRS-FM only": (0.00, 0.78),
        "Watanabe + scDRS only": (-0.48, -0.35),
        "scDRS-FM + scDRS only": (0.48, -0.35),
        "All three": (0.00, 0.02),
    }
    for region, position in count_positions.items():
        overlap_axis.text(
            *position,
            str(exact_regions[region]),
            ha="center",
            va="center",
            fontsize=18,
            fontweight="bold",
        )

    overlap_axis.text(
        -1.22, 1.28, f"Watanabe\n(n={len(watanabe_pairs)})",
        ha="center", va="center", fontsize=15, fontweight="bold",
    )
    overlap_axis.text(
        1.22, 1.28, f"scDRS-FM\n(n={len(scdrsfm_pairs)})",
        ha="center", va="center", fontsize=15, fontweight="bold",
    )
    overlap_axis.text(
        0.00, -1.74, f"scDRS\n(n={len(scdrs_pairs)})",
        ha="center", va="center", fontsize=15, fontweight="bold",
    )
    overlap_axis.text(
        0.00,
        -2.02,
        "Exact trait × cell-type counts; circle areas are not proportional",
        ha="center",
        fontsize=10.5,
    )
    overlap_axis.set_xlim(-1.8, 1.8)
    overlap_axis.set_ylim(-2.12, 1.7)
    overlap_axis.set_aspect("equal")
    overlap_axis.axis("off")

    plot_table = (
        method_watanabe_enrichment
        .set_index("method")
        .loc[["scDRS-FM", "scDRS"]]
        .reset_index()
    )
    y_positions = np.arange(len(plot_table))[::-1]
    method_colors = {"scDRS-FM": "#2a9d8f", "scDRS": "#457b9d"}

    for y_position, row in zip(y_positions, plot_table.itertuples(index=False)):
        odds_ratio = row.continuity_corrected_odds_ratio
        ci_low = row.odds_ratio_ci95_low
        ci_high = row.odds_ratio_ci95_high
        odds_ratio_axis.errorbar(
            odds_ratio,
            y_position,
            xerr=np.asarray([[odds_ratio - ci_low], [ci_high - odds_ratio]]),
            fmt="o",
            color=method_colors[row.method],
            markersize=10,
            capsize=5,
            linewidth=2.0,
        )

        fisher_p = row.fisher_one_sided_enrichment_pvalue
        p_text = f"{fisher_p:.2e}" if fisher_p < 0.001 else f"{fisher_p:.3f}"
        odds_ratio_axis.text(
            ci_high * 1.08,
            y_position,
            (
                f"OR={odds_ratio:.2f}  95% CI [{ci_low:.2f}, {ci_high:.2f}]\n"
                f"Fisher p={p_text}; enrichment={row.fold_enrichment:.2f}×"
            ),
            va="center",
            fontsize=11.5,
            clip_on=False,
        )

    odds_ratio_axis.axvline(1.0, color="#666666", linestyle="--", linewidth=1.5)
    odds_ratio_axis.set_xscale("log")
    odds_ratio_axis.set_yticks(y_positions)
    odds_ratio_axis.set_yticklabels(plot_table["method"], fontsize=14)
    odds_ratio_axis.set_xlabel("Odds ratio for overlap with Watanabe", fontsize=13)
    odds_ratio_axis.tick_params(axis="x", labelsize=11)
    odds_ratio_axis.grid(axis="x", color="#e9ecef", linewidth=0.8)
    odds_ratio_axis.set_ylim(-0.7, len(plot_table) - 0.3)
    odds_ratio_axis.set_xlim(
        max(float(plot_table["odds_ratio_ci95_low"].min()) / 2, 1e-3),
        float(plot_table["odds_ratio_ci95_high"].max()) * 7,
    )

    figure.tight_layout()
    return figure


final_figure = plot_final_comparison()
final_png = FIGURE_DIR / "figure3_three_method_overlap_enrichment_trait_bootstrap.png"
final_pdf = FIGURE_DIR / "figure3_three_method_overlap_enrichment_trait_bootstrap.pdf"
final_figure.savefig(final_png, dpi=FIGURE_DPI, bbox_inches="tight")
final_figure.savefig(final_pdf, bbox_inches="tight")
plt.show()
print(f"Saved: {final_png}\nSaved: {final_pdf}")


Saved: watanabe_unified_outputs/figures/figure3_three_method_overlap_enrichment_trait_bootstrap.png
Saved: watanabe_unified_outputs/figures/figure3_three_method_overlap_enrichment_trait_bootstrap.pdf


## 10. Manuscript supplementary CSV exports

This section writes tidy source-data tables for the figures and additional reproducibility tables. The full trait-by-cell-type universe is retained so zeros and non-overlaps are explicit. A manifest describes every CSV, and all CSV files are bundled into a single ZIP archive.


In [10]:
from zipfile import ZIP_DEFLATED, ZipFile


def _cell_count(cell_type: object) -> int:
    return int(CELLTYPE_COUNTS.get(str(cell_type), 0))


def _overlap_region(
    watanabe: bool,
    scdrsfm: bool,
    scdrs: bool,
) -> str:
    labels = {
        (True, False, False): "Watanabe only",
        (False, True, False): "scDRS-FM only",
        (False, False, True): "scDRS only",
        (True, True, False): "Watanabe + scDRS-FM only",
        (True, False, True): "Watanabe + scDRS only",
        (False, True, True): "scDRS-FM + scDRS only",
        (True, True, True): "All three",
        (False, False, False): "No source",
    }
    return labels[(bool(watanabe), bool(scdrsfm), bool(scdrs))]


def _calls_table(
    calls_by_trait: Mapping[str, Iterable[str]],
    method: str,
) -> pd.DataFrame:
    rows = [
        {
            "method": method,
            "trait": trait,
            "trait_label": TRAIT_LABELS[trait],
            "cell_type": cell_type,
            "cell_type_n_cells": _cell_count(cell_type),
        }
        for trait in TRAITS
        for cell_type in sorted(set(calls_by_trait.get(trait, [])))
    ]
    return pd.DataFrame(
        rows,
        columns=[
            "method", "trait", "trait_label", "cell_type", "cell_type_n_cells"
        ],
    )


# -----------------------------------------------------------------------------
# TMS annotation and Watanabe source tables
# -----------------------------------------------------------------------------
cell_type_counts_table = (
    CELLTYPE_COUNTS
    .rename("n_cells")
    .rename_axis("cell_type")
    .reset_index()
)
cell_type_counts_table["fraction_of_all_cells"] = (
    cell_type_counts_table["n_cells"] / cell_type_counts_table["n_cells"].sum()
)
cell_type_counts_table["percent_of_all_cells"] = (
    100.0 * cell_type_counts_table["fraction_of_all_cells"]
)
cell_type_counts_table["rank_by_cell_count"] = (
    cell_type_counts_table["n_cells"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
cell_type_counts_table = cell_type_counts_table.sort_values(
    ["n_cells", "cell_type"], ascending=[False, True]
).reset_index(drop=True)

watanabe_reference_rows: list[dict[str, object]] = []
for trait in TRAITS:
    for raw_cell_type in WATANABE_CELL_TYPES_RAW[trait]:
        mapped_cell_type, in_tms_universe = map_celltype_label(
            raw_cell_type,
            CELLTYPE_UNIVERSE,
            CELLTYPE_ALIASES,
        )
        watanabe_reference_rows.append({
            "trait": trait,
            "trait_label": TRAIT_LABELS[trait],
            "raw_watanabe_cell_type": raw_cell_type,
            "harmonized_cell_type": mapped_cell_type,
            "mapped_to_tms_universe": bool(in_tms_universe),
            "cell_type_n_cells": _cell_count(mapped_cell_type) if in_tms_universe else 0,
        })
watanabe_reference_table = pd.DataFrame(watanabe_reference_rows)

# -----------------------------------------------------------------------------
# scDRS-FM and scDRS call/source tables
# -----------------------------------------------------------------------------
scdrsfm_fraction_long = (
    scdrsfm_marginal_x_conditional_fractions
    .rename_axis(index="trait", columns="cell_type")
    .reset_index()
    .melt(
        id_vars="trait",
        var_name="cell_type",
        value_name="marginal_x_conditional_discovery_fraction",
    )
)
scdrsfm_fraction_long.insert(
    1,
    "trait_label",
    scdrsfm_fraction_long["trait"].map(TRAIT_LABELS),
)
scdrsfm_fraction_long["cell_type_n_cells"] = (
    scdrsfm_fraction_long["cell_type"].map(CELLTYPE_COUNTS).fillna(0).astype(int)
)
scdrsfm_fraction_long["celltype_call_threshold_strictly_greater_than"] = (
    CELLTYPE_CALL_THRESHOLD
)
scdrsfm_fraction_long["scdrsfm_called"] = scdrsfm_fraction_long[
    "marginal_x_conditional_discovery_fraction"
].gt(CELLTYPE_CALL_THRESHOLD)

scdrsfm_fraction_wide = (
    scdrsfm_marginal_x_conditional_fractions
    .rename_axis(index="trait", columns="cell_type")
    .reset_index()
)
scdrsfm_fraction_wide.insert(
    1,
    "trait_label",
    scdrsfm_fraction_wide["trait"].map(TRAIT_LABELS),
)

scdrsfm_calls_table = scdrsfm_fraction_long.loc[
    scdrsfm_fraction_long["scdrsfm_called"]
].copy()
scdrs_calls_table = _calls_table(SCDRS_BY_TRAIT, "scDRS")

method_calls_combined = pd.concat([
    _calls_table(WATANABE_MAPPED_BY_TRAIT, "Watanabe").assign(
        call_rule="Published Watanabe reference association"
    ),
    _calls_table(SCDRSFM_BY_TRAIT, "scDRS-FM").assign(
        call_rule=(
            f"marginal x conditional discovery fraction > {CELLTYPE_CALL_THRESHOLD:g}"
        )
    ),
    scdrs_calls_table.assign(
        call_rule=f"BH-FDR <= {SCDRS_FDR:g}"
    ),
], ignore_index=True)

signal_sharing_detail = build_signal_sharing_detail()
if not signal_sharing_detail.empty:
    signal_sharing_detail.insert(
        1,
        "trait_label",
        signal_sharing_detail["trait"].map(TRAIT_LABELS),
    )
    signal_sharing_detail["cell_type_n_cells"] = (
        signal_sharing_detail["cell_type"].map(CELLTYPE_COUNTS).fillna(0).astype(int)
    )

signal_association_table = scdrsfm_signal_celltype_associations.copy()
if not signal_association_table.empty:
    signal_association_table.insert(
        1,
        "trait_label",
        signal_association_table["trait"].map(TRAIT_LABELS),
    )
    signal_association_table["cell_type_n_cells"] = (
        signal_association_table["cell_type"].map(CELLTYPE_COUNTS).fillna(0).astype(int)
    )

signal_summary = (
    signal_association_table
    .groupby(["trait", "cell_type"], as_index=False)
    .agg(
        scdrsfm_independent_signals=(
            "independent_signal",
            lambda values: ";".join(map(str, sorted(set(map(int, values))))),
        ),
        n_scdrsfm_independent_signals=("independent_signal", "nunique"),
        max_signal_discovery_fraction=("signal_discovery_fraction", "max"),
    )
    if not signal_association_table.empty
    else pd.DataFrame(columns=[
        "trait", "cell_type", "scdrsfm_independent_signals",
        "n_scdrsfm_independent_signals", "max_signal_discovery_fraction",
    ])
)

# -----------------------------------------------------------------------------
# Full trait x cell-type membership table and exact overlap regions
# -----------------------------------------------------------------------------
membership_rows: list[dict[str, object]] = []
for trait in TRAITS:
    for cell_type in CELLTYPE_UNIVERSE:
        is_watanabe = cell_type in watanabe_by_trait[trait]
        is_scdrsfm = cell_type in scdrsfm_by_trait[trait]
        is_scdrs = cell_type in scdrs_by_trait[trait]
        membership_rows.append({
            "trait": trait,
            "trait_label": TRAIT_LABELS[trait],
            "cell_type": cell_type,
            "cell_type_n_cells": _cell_count(cell_type),
            "watanabe_reference": is_watanabe,
            "scdrsfm_called": is_scdrsfm,
            "scdrs_called": is_scdrs,
            "n_supporting_sources": int(is_watanabe) + int(is_scdrsfm) + int(is_scdrs),
            "exact_overlap_region": _overlap_region(
                is_watanabe,
                is_scdrsfm,
                is_scdrs,
            ),
        })
trait_celltype_membership = pd.DataFrame(membership_rows)
trait_celltype_membership = trait_celltype_membership.merge(
    scdrsfm_fraction_long[[
        "trait", "cell_type", "marginal_x_conditional_discovery_fraction"
    ]],
    on=["trait", "cell_type"],
    how="left",
)
trait_celltype_membership = trait_celltype_membership.merge(
    signal_summary,
    on=["trait", "cell_type"],
    how="left",
)
trait_celltype_membership["n_scdrsfm_independent_signals"] = (
    trait_celltype_membership["n_scdrsfm_independent_signals"]
    .fillna(0)
    .astype(int)
)
trait_celltype_membership["scdrsfm_independent_signals"] = (
    trait_celltype_membership["scdrsfm_independent_signals"].fillna("")
)
trait_celltype_supported_pairs = trait_celltype_membership.loc[
    trait_celltype_membership["n_supporting_sources"].gt(0)
].reset_index(drop=True)

region_order = [
    "Watanabe only",
    "scDRS-FM only",
    "scDRS only",
    "Watanabe + scDRS-FM only",
    "Watanabe + scDRS only",
    "scDRS-FM + scDRS only",
    "All three",
    "No source",
]
exact_overlap_counts_table = (
    trait_celltype_membership["exact_overlap_region"]
    .value_counts()
    .reindex(region_order, fill_value=0)
    .rename("n_trait_celltype_pairs")
    .rename_axis("exact_overlap_region")
    .reset_index()
)
exact_overlap_counts_table["fraction_of_trait_celltype_universe"] = (
    exact_overlap_counts_table["n_trait_celltype_pairs"]
    / len(trait_celltype_membership)
)

# -----------------------------------------------------------------------------
# Per-trait and pooled enrichment tables
# -----------------------------------------------------------------------------
per_trait_enrichment_rows: list[dict[str, object]] = []
for method, trait_counts in counts_by_method_trait.items():
    for trait in TRAITS:
        row = summarize_enrichment(method, trait_counts[trait])
        row.update({
            "trait": trait,
            "trait_label": TRAIT_LABELS[trait],
            "n_cell_types_in_universe": N_CELL_TYPES,
        })
        per_trait_enrichment_rows.append(row)
per_trait_watanabe_enrichment = pd.DataFrame(per_trait_enrichment_rows)
per_trait_watanabe_enrichment = per_trait_watanabe_enrichment[[
    "method", "trait", "trait_label", "n_cell_types_in_universe",
    "overlap", "method_calls", "watanabe_calls", "method_only",
    "watanabe_only", "neither", "expected_overlap", "fold_enrichment",
    "fisher_odds_ratio", "continuity_corrected_odds_ratio",
    "odds_ratio_ci95_low", "odds_ratio_ci95_high", "log_odds_ratio",
    "fisher_one_sided_enrichment_pvalue",
]]

pooled_method_enrichment = method_watanabe_enrichment.copy()
pooled_method_enrichment.insert(1, "n_traits", len(TRAITS))
pooled_method_enrichment.insert(2, "n_cell_types_per_trait", N_CELL_TYPES)
pooled_method_enrichment.insert(3, "n_trait_celltype_pairs", len(TRAITS) * N_CELL_TYPES)

bootstrap_replicates_table = pd.DataFrame({
    "bootstrap_replicate": np.arange(1, BOOTSTRAP_REPS + 1, dtype=int),
    "scDRS-FM_fold_enrichment": fm_bootstrap_fold,
    "scDRS_fold_enrichment": scdrs_bootstrap_fold,
    "scDRS-FM_minus_scDRS": bootstrap_difference,
})
bootstrap_replicates_table["scDRS-FM_greater_than_scDRS"] = (
    bootstrap_replicates_table["scDRS-FM_minus_scDRS"].gt(0)
)
bootstrap_replicates_table["seed"] = BOOTSTRAP_SEED
bootstrap_replicates_table["n_traits_resampled"] = len(TRAITS)

analysis_parameters = pd.DataFrame([
    {"parameter": "tms_annotation_source", "value": TMS_ANNOTATION_SOURCE},
    {"parameter": "scdrsfm_results_directory", "value": str(SCDRSFM_RESULTS_DIR.resolve())},
    {"parameter": "scdrs_results_directory", "value": str(SCDRS_RESULTS_DIR.resolve())},
    {"parameter": "cell_type_annotation_column", "value": BIOCOL},
    {"parameter": "minimum_genes_per_cell", "value": FILTER_CELLS_MIN_GENES},
    {"parameter": "n_cells_after_filtering", "value": len(obs)},
    {"parameter": "n_cell_types", "value": N_CELL_TYPES},
    {"parameter": "n_traits", "value": len(TRAITS)},
    {"parameter": "scdrsfm_fdr", "value": SCDRSFM_FDR},
    {"parameter": "scdrs_fdr", "value": SCDRS_FDR},
    {"parameter": "celltype_call_threshold_strictly_greater_than", "value": CELLTYPE_CALL_THRESHOLD},
    {"parameter": "signal_association_threshold_greater_than_or_equal_to", "value": SIGNAL_ASSOCIATION_THRESHOLD},
    {"parameter": "bootstrap_replicates", "value": BOOTSTRAP_REPS},
    {"parameter": "bootstrap_seed", "value": BOOTSTRAP_SEED},
])

# -----------------------------------------------------------------------------
# Write manuscript-facing CSVs and a manifest
# -----------------------------------------------------------------------------
csv_specs = [
    (
        "tms_cell_type_counts.csv",
        cell_type_counts_table,
        "TMS cell counts, fractions, percentages, and ranks for every cell type.",
    ),
    (
        "watanabe_reference_associations.csv",
        watanabe_reference_table,
        "Published Watanabe trait-cell-type associations with TMS label harmonization.",
    ),
    (
        "scdrsfm_celltype_discovery_fractions_all.csv",
        scdrsfm_fraction_long,
        "scDRS-FM marginal-by-conditional discovery fraction for every trait-cell-type pair.",
    ),
    (
        "scdrsfm_celltype_discovery_fractions_wide.csv",
        scdrsfm_fraction_wide,
        "Wide trait-by-cell-type matrix of scDRS-FM marginal-by-conditional discovery fractions.",
    ),
    (
        "method_celltype_calls_combined.csv",
        method_calls_combined,
        "Combined Watanabe, scDRS-FM, and scDRS trait-cell-type call list with call rules.",
    ),
    (
        "scdrsfm_celltype_calls.csv",
        scdrsfm_calls_table,
        "scDRS-FM trait-cell-type calls passing the strict discovery-fraction threshold.",
    ),
    (
        "scdrs_celltype_calls.csv",
        scdrs_calls_table,
        "scDRS trait-cell-type calls passing the configured BH-FDR threshold.",
    ),
    (
        "scdrsfm_signal_celltype_associations.csv",
        signal_association_table,
        "Significant scDRS-FM independent-signal to cell-type associations.",
    ),
    (
        "scdrsfm_signal_sharing_detail.csv",
        signal_sharing_detail,
        "Signal-sharing status between scDRS-FM calls and direct Watanabe-overlap anchors.",
    ),
    (
        "figure1_scdrsfm_watanabe_celltype_signal_overlap.csv",
        figure1_table,
        "Source data for the Watanabe/scDRS-FM cell-type and signal-overlap matrix.",
    ),
    (
        "trait_celltype_method_membership_full.csv",
        trait_celltype_membership,
        "Complete trait-by-cell-type universe with Watanabe, scDRS-FM, and scDRS membership.",
    ),
    (
        "trait_celltype_supported_pairs.csv",
        trait_celltype_supported_pairs,
        "Trait-cell-type pairs supported by at least one of Watanabe, scDRS-FM, or scDRS.",
    ),
    (
        "three_method_exact_overlap_counts.csv",
        exact_overlap_counts_table,
        "Exact three-source overlap-region counts, including unsupported pairs.",
    ),
    (
        "watanabe_enrichment_by_trait_and_method.csv",
        per_trait_watanabe_enrichment,
        "Per-trait contingency counts, odds ratios, p-values, and fold enrichment versus Watanabe.",
    ),
    (
        "watanabe_enrichment_pooled_by_method.csv",
        pooled_method_enrichment,
        "Pooled Watanabe enrichment statistics for scDRS-FM and scDRS.",
    ),
    (
        "fold_enrichment_scdrsfm_vs_scdrs_trait_bootstrap.csv",
        fold_enrichment_comparison,
        "Observed paired trait-bootstrap comparison of scDRS-FM and scDRS fold enrichment.",
    ),
    (
        "fold_enrichment_trait_bootstrap_replicates.csv",
        bootstrap_replicates_table,
        "All paired trait-bootstrap fold-enrichment replicates and method differences.",
    ),
    (
        "analysis_parameters_and_sources.csv",
        analysis_parameters,
        "Configured inputs, thresholds, dimensions, and random seed used by the analysis.",
    ),
]

manifest_rows: list[dict[str, object]] = []
for filename, frame, description in csv_specs:
    relative_path = f"tables/csv/{filename}"
    saved_path = save_table(
        frame,
        relative_path,
        also_csv=False,
        index=False,
    )
    manifest_rows.append({
        "file": filename,
        "relative_path": str(saved_path.relative_to(OUTPUT_DIR)),
        "description": description,
        "n_rows": int(len(frame)),
        "n_columns": int(len(frame.columns)),
        "columns": ";".join(map(str, frame.columns)),
    })

csv_manifest = pd.DataFrame(manifest_rows).sort_values("file").reset_index(drop=True)
manifest_path = save_table(
    csv_manifest,
    "tables/csv/csv_manifest.csv",
    also_csv=False,
    index=False,
)

zip_path = OUTPUT_DIR / "watanabe_manuscript_supplementary_csvs.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    for csv_path in sorted(CSV_DIR.rglob("*.csv")):
        archive.write(csv_path, arcname=csv_path.relative_to(CSV_DIR))

print(f"Saved CSV manifest: {manifest_path}")
print(f"Saved CSV bundle:   {zip_path}")
print(
    f"Exported {len(csv_manifest):,} documented source-data tables "
    f"plus the manifest to {CSV_DIR.resolve()}"
)
display(csv_manifest)


Saved: watanabe_unified_outputs/tables/csv/tms_cell_type_counts.csv
Saved: watanabe_unified_outputs/tables/csv/watanabe_reference_associations.csv


Saved: watanabe_unified_outputs/tables/csv/scdrsfm_celltype_discovery_fractions_all.csv
Saved: watanabe_unified_outputs/tables/csv/scdrsfm_celltype_discovery_fractions_wide.csv
Saved: watanabe_unified_outputs/tables/csv/method_celltype_calls_combined.csv
Saved: watanabe_unified_outputs/tables/csv/scdrsfm_celltype_calls.csv
Saved: watanabe_unified_outputs/tables/csv/scdrs_celltype_calls.csv
Saved: watanabe_unified_outputs/tables/csv/scdrsfm_signal_celltype_associations.csv
Saved: watanabe_unified_outputs/tables/csv/scdrsfm_signal_sharing_detail.csv
Saved: watanabe_unified_outputs/tables/csv/figure1_scdrsfm_watanabe_celltype_signal_overlap.csv
Saved: watanabe_unified_outputs/tables/csv/trait_celltype_method_membership_full.csv
Saved: watanabe_unified_outputs/tables/csv/trait_celltype_supported_pairs.csv
Saved: watanabe_unified_outputs/tables/csv/three_method_exact_overlap_counts.csv
Saved: watanabe_unified_outputs/tables/csv/watanabe_enrichment_by_trait_and_method.csv
Saved: watanabe_uni

Saved: watanabe_unified_outputs/tables/csv/fold_enrichment_trait_bootstrap_replicates.csv
Saved: watanabe_unified_outputs/tables/csv/analysis_parameters_and_sources.csv
Saved: watanabe_unified_outputs/tables/csv/csv_manifest.csv


Saved CSV manifest: watanabe_unified_outputs/tables/csv/csv_manifest.csv
Saved CSV bundle:   watanabe_unified_outputs/watanabe_manuscript_supplementary_csvs.zip
Exported 18 documented source-data tables plus the manifest to /workspace/nbwork/06_watanabe_overlap/watanabe_unified_outputs/tables/csv


,file,relative_path,description,n_rows,n_columns,columns
0,analysis_parameters_and_sources.csv,tables/csv/analysis_parameters_and_sources.csv,"Configured inputs, thresholds, dimensions, and...",14,2,parameter;value
1,figure1_scdrsfm_watanabe_celltype_signal_overl...,tables/csv/figure1_scdrsfm_watanabe_celltype_s...,Source data for the Watanabe/scDRS-FM cell-typ...,117,12,trait;trait_label;cell_type;watanabe_listed;wa...
2,fold_enrichment_scdrsfm_vs_scdrs_trait_bootstr...,tables/csv/fold_enrichment_scdrsfm_vs_scdrs_tr...,Observed paired trait-bootstrap comparison of ...,1,11,comparison;scDRS-FM_fold_enrichment;scDRS_fold...
3,fold_enrichment_trait_bootstrap_replicates.csv,tables/csv/fold_enrichment_trait_bootstrap_rep...,All paired trait-bootstrap fold-enrichment rep...,20000,7,bootstrap_replicate;scDRS-FM_fold_enrichment;s...
4,method_celltype_calls_combined.csv,tables/csv/method_celltype_calls_combined.csv,"Combined Watanabe, scDRS-FM, and scDRS trait-c...",242,6,method;trait;trait_label;cell_type;cell_type_n...
5,scdrs_celltype_calls.csv,tables/csv/scdrs_celltype_calls.csv,scDRS trait-cell-type calls passing the config...,96,5,method;trait;trait_label;cell_type;cell_type_n...
6,scdrsfm_celltype_calls.csv,tables/csv/scdrsfm_celltype_calls.csv,scDRS-FM trait-cell-type calls passing the str...,105,7,trait;trait_label;cell_type;marginal_x_conditi...
7,scdrsfm_celltype_discovery_fractions_all.csv,tables/csv/scdrsfm_celltype_discovery_fraction...,scDRS-FM marginal-by-conditional discovery fra...,1160,7,trait;trait_label;cell_type;marginal_x_conditi...
8,scdrsfm_celltype_discovery_fractions_wide.csv,tables/csv/scdrsfm_celltype_discovery_fraction...,Wide trait-by-cell-type matrix of scDRS-FM mar...,10,118,trait;trait_label;B cell;Bergmann glial cell;B...
9,scdrsfm_signal_celltype_associations.csv,tables/csv/scdrsfm_signal_celltype_association...,Significant scDRS-FM independent-signal to cel...,93,6,trait;trait_label;cell_type;independent_signal...
